In [3]:
!pip install -q \
sentence-transformers \
faiss-cpu \
langchain-community \
langchain-text-splitters \
transformers \
pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
from google.colab import files
uploaded = files.upload()

Saving intro-to-ml.pdf to intro-to-ml (1).pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/intro-to-ml.pdf")
documents = loader.load()

print("Total Pages:", len(documents))
print(documents[0].page_content[:500])

Total Pages: 392
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 1137


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc.page_content for doc in chunks]

embeddings = model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Embeddings shape: (1137, 384)


In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS index ready")

FAISS index ready


In [9]:
def retrieve(query, k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [texts[i] for i in indices[0]]


In [11]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

print("LLM loaded successfully")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

LLM loaded successfully


In [12]:
def rag(query, k=5):
    docs = retrieve(query, k)
    context = "\n\n".join(docs)

    prompt = f"""
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    return llm(prompt)[0]['generated_text']

In [13]:
print(rag("What is supervised learning?"))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We already saw an application of supervised machine learning in Chapter 1:
classifying iris flowers into several species using physical measurements of the
flowers.
Remember that supervised learning is used whenever we want to predict a certain
outcome from a given input, and we have examples of input/output pairs. We build a
machine learning model from these input/output pairs, which comprise our training
set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐

2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

In [14]:
queries = [
    "What is supervised learning?",
    "Explain overfitting",
    "What is classification?",
    "What is feature engineering?",
    "Explain k-nearest neighbors"
]

for q in queries:
    print("\n====================")
    print("Q:", q)
    print("A:", rag(q))


Q: What is supervised learning?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Token indices sequence length is longer than the specified maximum sequence length for this model (975 > 512). Running this sequence through the model will result in indexing errors


A: 
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We already saw an application of supervised machine learning in Chapter 1:
classifying iris flowers into several species using physical measurements of the
flowers.
Remember that supervised learning is used whenever we want to predict a certain
outcome from a given input, and we have examples of input/output pairs. We build a
machine learning model from these input/output pairs, which comprise our training
set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐

2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . .

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
the leaf the new data point falls into. The output for this data point is the mean target
of the training points in this leaf.
Controlling complexity of decision trees
Typically, building a tree as described here and continuing until all leaves are pure
leads to models that are very complex and highly overfit to the training data. The
presence of pure leaves mean that a tree is 100% accurate on the training set; each
data point in the training set is in a leaf that has the correct majority class. The over‐
fitting can be seen on the left of Figure 2-26. Y ou can see the regions determined to
belong to class 1 in the middle of all the points belonging to class 0. On the other
hand, there is a small strip predicted as class 0 around the point belonging to class 0

on a single customer.
The only measure of whether an algorithm will perform well on new data i

...

A: 
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
which is the special case of distinguishing between exactly two classes, and multiclass
classification, which is classification between more than two classes. Y ou can think of
binary classification as trying to answer a yes/no question. Classifying emails as
either spam or not spam is an example of a binary classification problem. In this
binary classification task, the yes/no question being asked would be “Is this email
spam?”
25

set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐
vised learning often requires human effort to build the training set, but afterward
automates and often speeds up an otherwise laborious or infeasible task.
Classification  and Regression
There are two major types of supervised machine learning problems, called classifica‐
tion and regression.
In classification, the goal is to predict a class

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
CHAPTER 4
Representing Data and
Engineering Features
So far, we’ve assumed that our data comes in as a two-dimensional array of floating-
point numbers, where each column is a continuous feature that describes the data
points. For many applications, this is not how the data is collected. A particularly
common type of feature is the categorical features. Also known as discrete features,
these are usually not numeric. The distinction between categorical features and con‐
tinuous features is analogous to the distinction between classification and regression,
only on the input side rather than the output side. Examples of continuous features
that we have seen are pixel brightnesses and size measurements of plant flowers.

rithms, automatic feature selection can be quite helpful. It is also great for reducing
the amount of features needed—for example, to speed

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
You are a Machine Learning assistant.

Answer ONLY using the context below.
If not found, say "Not found in document".

Context:
The k in k-nearest neighbors signifies that instead of using only the closest neighbor
to the new data point, we can consider any fixed number k of neighbors in the train‐
ing (for example, the closest three or five neighbors). Then, we can make a prediction
using the majority class among these neighbors. We will go into more detail about
this in Chapter 2; for now, we’ll use only a single neighbor.
All machine learning models in scikit-learn are implemented in their own classes,
which are called Estimator classes. The k-nearest neighbors classification algorithm
is implemented in the KNeighborsClassifier class in the neighbors module. Before
we can use the model, we need to instantiate the class into an object. This is when we

Again, the prediction is shown as the color of the cross. Y ou can see that the predic‐
tion for the new data point at the top l